# TZ-SAD — batch run

Trustworthy zero-shot anomaly detection on MVTec-AD: uncertainty quantification,
conformal calibration, hallucination detection and the evaluation protocol.

**This notebook is self-contained.** The code is embedded below, so it needs no
repository access. Enable **GPU** and **Internet**, then *Save Version →
Save & Run All (Commit)*, or schedule it.

**To continue a previous run**, add that run's notebook output as an input dataset
(*Add data → Your Work → Notebook Output*). Every expensive stage checkpoints, so
the next run resumes instead of restarting.


In [ ]:
# --- 1. Configuration -------------------------------------------------------
PHASES = "clip,synthetic,bakeoff,corruption"           # clip,synthetic,bakeoff,corruption,qwen,hallucination,fusion
BUDGET_MINUTES = 480     # Kaggle kills the run at 9h (GPU) / 12h (CPU); stay under it
RESERVE_MINUTES = 12          # held back so the final phase can write its outputs
PREV_OUTPUT = ""     # input dataset holding a previous run's /kaggle/working, or ""

# Where run outputs go. Override via TZSAD_RESULTS to keep a test run from writing
# into a directory that already holds full-scale results.
import os
RESULTS = os.environ.get("TZSAD_RESULTS", "/kaggle/working/results")

# Extra dotlist overrides applied to every phase, e.g.
# OVERRIDES = ["data.categories=[bottle,carpet]", "data.subset=n_per_folder=10"]
OVERRIDES = []
print("phases:", PHASES, "| budget:", BUDGET_MINUTES, "min")

In [ ]:
# --- 2. Environment ---------------------------------------------------------
import os, subprocess, sys, time, shutil, pathlib
T0 = time.time()

def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip()[-2000:])
    if check and r.returncode: raise RuntimeError(r.stderr[-2000:])
    return r.returncode

print(sh("nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader") or "")
sh("pip install -q open_clip_torch qwen-vl-utils omegaconf 2>&1 | tail -2")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpus", torch.cuda.device_count())

In [ ]:
# --- 3. Restore any previous run's state -----------------------------------
# A committed notebook saves /kaggle/working as its output. Attaching that output
# to the next run is what makes the pipeline resumable across scheduled runs.
os.makedirs(RESULTS, exist_ok=True)
print("results ->", RESULTS)

restored = 0
if PREV_OUTPUT:
    src = f"/kaggle/input/{PREV_OUTPUT}/results"
    if os.path.isdir(src):
        for entry in os.listdir(src):
            dst = os.path.join(RESULTS, entry)
            if not os.path.exists(dst):
                shutil.copytree(os.path.join(src, entry), dst, dirs_exist_ok=True)
                restored += 1
        print(f"restored {restored} run directories from {src}")
    else:
        print(f"no previous output at {src} (starting fresh)")
else:
    print("PREV_OUTPUT not set - starting fresh")

led = pathlib.Path(RESULTS) / "run_all" / "phases.json"
print("phases already complete:", list(__import__("json").loads(led.read_text()))
      if led.exists() else "none")

In [ ]:
# --- 4. Unpack the embedded code -------------------------------------------
import base64, io, tarfile
CODE_B64 = "..."

ROOT = "/kaggle/working/tzsad"
os.makedirs(ROOT, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64))) as t:
    t.extractall(ROOT)          # overwrite in place; never delete a live tree
os.chdir(ROOT)
print("code unpacked ->", ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# --- 5. Locate the dataset --------------------------------------------------
# Attach MVTec-AD (e.g. the "ipythonx/mvtec-ad" dataset) via Add data, or let
# kagglehub fetch it when Internet is enabled.
sys.path.insert(0, ROOT)
from tzsad.utils.env import resolve_data_root

try:
    DATA_ROOT = resolve_data_root("auto")
except FileNotFoundError:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("ipythonx/mvtec-ad")
print("data root:", DATA_ROOT)
print(sorted(os.listdir(DATA_ROOT))[:8])

In [ ]:
# --- 6. Self-test before spending GPU quota --------------------------------
# The suite runs on stubbed backbones in about a minute. If it fails, the batch
# run would waste hours producing nothing, so stop here instead.
rc = sh("python -m pytest -q --tb=line -p no:warnings 2>&1 | tail -6")
assert rc == 0, "test suite failed - not starting the batch run"
print("\nself-test passed")

In [ ]:
# --- 7. Run the pipeline ----------------------------------------------------
elapsed_min = (time.time() - T0) / 60
budget = max(BUDGET_MINUTES - elapsed_min, 5)
cmd = (f"python scripts/run_all.py --budget-minutes {budget:.0f} "
       f"--reserve-minutes {RESERVE_MINUTES} --phases {PHASES} "
       f"data.root={DATA_ROOT} paths.results={RESULTS} paths.cache={RESULTS}/cache "
       + " ".join(OVERRIDES))
print(cmd, flush=True)

proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:            # stream, so a killed run still shows progress
    print(line, end="", flush=True)
rc = proc.wait()
print("\nrun_all exit code:", rc, "(0 complete, 2 partial - re-run to continue)")

In [ ]:
# --- 8. Audit the outputs ---------------------------------------------------
import pandas as pd, glob, json
pd.set_option("display.width", 200)

runs = sorted(d for d in glob.glob(f"{RESULTS}/*") if os.path.isfile(f"{d}/manifest.json"))
print("run directories:", [os.path.basename(r) for r in runs])

for name, pattern in [
    ("detection",        f"{RESULTS}/*/report/detection.csv"),
    ("bake-off",         f"{RESULTS}/*/report/bakeoff_error_prediction.csv"),
    ("corruption",       f"{RESULTS}/*/report/corruption_monotonicity.csv"),
    ("coverage",         f"{RESULTS}/*/report/coverage_sweep.csv"),
]:
    for path in sorted(glob.glob(pattern)):
        df = pd.read_csv(path)
        print(f"\n=== {name}: {path.replace(RESULTS + '/', '')} ({len(df)} rows) ===")
        if name == "detection":
            print(df[df.group.isin(["POOLED", "MEAN_OF_CATEGORIES"])].to_string(index=False))
        elif name == "bake-off":
            sub = df[(df.scope == "pred=0") & (df.half == "all")]
            print(sub.sort_values("err_auroc", ascending=False)[
                ["signal", "err_auroc", "ci_lo", "ci_hi", "informative", "misleading"]
            ].to_string(index=False))
        elif name == "corruption":
            print(df.groupby("signal").agg(
                silent=("silent_failure", "sum"),
                mean_rho=("spearman_severity_vs_uncertainty", "mean")).to_string())
        else:
            print(df[df.group == "POOLED"][
                ["delta", "nominal", "empirical_coverage", "ci_lo", "ci_hi"]
            ].to_string(index=False))

In [ ]:
# --- 9. Sanity checks -------------------------------------------------------
# Cheap invariants that catch a corrupted run before the numbers are believed.
problems = []
for path in sorted(glob.glob(f"{RESULTS}/*/records_test_scored.parquet")):
    df = pd.read_parquet(path)
    tag = path.replace(RESULTS + "/", "")
    if not df["anomaly_score"].dropna().between(0, 1).all():
        problems.append(f"{tag}: anomaly_score outside [0,1]")
    if df["image_id"].duplicated().any():
        problems.append(f"{tag}: duplicate image_id")
    if "conformal_p" in df and not df["conformal_p"].dropna().between(0, 1).all():
        problems.append(f"{tag}: conformal_p outside [0,1]")
    if "label" in df and not set(df["label"].unique()) <= {0, 1}:
        problems.append(f"{tag}: label not in {{0,1}}")
    print(f"{tag}: {len(df)} rows, {int(df.label.sum())} anomalous, "
          f"abstention {1 - df.parse_ok.mean():.3f}")

print("\n" + ("PROBLEMS:\n  " + "\n  ".join(problems) if problems else "all sanity checks passed"))
print(f"\ntotal notebook time: {(time.time() - T0) / 60:.1f} min")
print("outputs under /kaggle/working/results are saved as this version's output;")
print("attach them to the next run via PREV_OUTPUT to continue.")